# 03. Rekayasa Fitur

Tujuan notebook ini:
1. Menyusun kalender lengkap (data historis + bulan masa depan)
2. Membuat fitur kalender (Lebaran lag/lead, hari kerja, COVID)
3. Membuat regressor untuk SARIMAX
4. Membuat dataset LightGBM per horizon (1 sampai 6 bulan), tanpa kebocoran data
5. Menguji kebocoran data
6. Menyimpan hasil ke `data/processed/`

Strategi: **direct multi-horizon** (satu dataset dan satu model per horizon).

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath("../src"))

import config as cfg
import features as ft

FOLDER_PROCESSED = "../data/processed"
FOLDER_GAMBAR = "../reports/figures"
os.makedirs(FOLDER_GAMBAR, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

ModuleNotFoundError: No module named 'config'

In [ ]:
# data aktual (target + kalender)
df = pd.read_csv(os.path.join(FOLDER_PROCESSED, "data_final.csv"), parse_dates=["tanggal"])
df = df.set_index("tanggal").asfreq("MS")

# kalender bulan yang belum punya data aktual
future = pd.read_csv(os.path.join(FOLDER_PROCESSED, "kalender_future.csv"), parse_dates=["tanggal"])
future = future.set_index("tanggal")

print("data_final      :", df.shape, "|", df.index.min().strftime("%Y-%m"), "sampai", df.index.max().strftime("%Y-%m"))
print("kalender_future :", future.shape, "|", future.index.min().strftime("%Y-%m"), "sampai", future.index.max().strftime("%Y-%m"))

In [ ]:
kolom_kalender = ["total_hari", "jumlah_hari_kerja", "jumlah_libur_nasional_weekday",
                  "is_lebaran", "is_covid"]

kal_lengkap = pd.concat([df[kolom_kalender], future[kolom_kalender]]).sort_index()
kal_lengkap = kal_lengkap.asfreq("MS")

print("Kalender lengkap:", kal_lengkap.shape)
print("Rentang         :", kal_lengkap.index.min().strftime("%Y-%m"), "sampai", kal_lengkap.index.max().strftime("%Y-%m"))
print("Nilai kosong    :", kal_lengkap.isna().sum().sum())

In [ ]:
kal_fitur = ft.buat_fitur_kalender(kal_lengkap)

print(kal_fitur.head(8))
print()
print(kal_fitur.dtypes)

In [ ]:
# Di sekitar tiap bulan Lebaran: bulan sebelumnya harus lead1=1,
# bulan Lebaran is_lebaran=1, bulan sesudahnya lag1=1.
tanggal_lebaran = kal_fitur.index[kal_fitur["is_lebaran"] == 1]

for tgl in tanggal_lebaran[:4]:
    awal = tgl - pd.DateOffset(months=1)
    akhir = tgl + pd.DateOffset(months=1)
    print(kal_fitur.loc[awal:akhir, ["is_lebaran", "lebaran_lag1", "lebaran_lead1"]])
    print()

# setiap bulan Lebaran harus punya tepat satu tetangga lead dan satu lag
# (kecuali di ujung data)
print("Jumlah is_lebaran     :", kal_fitur["is_lebaran"].sum())
print("Jumlah lebaran_lag1   :", kal_fitur["lebaran_lag1"].sum())
print("Jumlah lebaran_lead1  :", kal_fitur["lebaran_lead1"].sum())

In [ ]:
exog = ft.buat_exog_sarimax(kal_fitur)

print("Regressor SARIMAX:", exog.shape)
print(exog.head())

# cek korelasi antar regressor (kolinearitas)
print("\nKorelasi antar regressor:")
print(exog.corr().round(2))

In [ ]:
df[cfg.KOLOM_TARGET_ALT] = df[cfg.KOLOM_TARGET] / df["jumlah_hari_kerja"]

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(df.index, df[cfg.KOLOM_TARGET] / 1000, color="steelblue")
ax[0].set_ylabel("Juta / bulan")
ax[0].set_title("Target utama: total penumpang per bulan")

ax[1].plot(df.index, df[cfg.KOLOM_TARGET_ALT], color="darkorange")
ax[1].set_ylabel("Ribu / hari kerja")
ax[1].set_title("Target alternatif: penumpang per hari kerja")
plt.tight_layout()
plt.savefig(os.path.join(FOLDER_GAMBAR, "fitur_01_target_alternatif.png"), dpi=150, bbox_inches="tight")
plt.show()

# Variasi (koefisien variasi) setelah COVID: target mana yang lebih "tenang"?
pasca = df[df.index >= cfg.PASCA_COVID]
for kolom in [cfg.KOLOM_TARGET, cfg.KOLOM_TARGET_ALT]:
    cv = pasca[kolom].std() / pasca[kolom].mean() * 100
    print(f"Koefisien variasi {kolom:28s}: {cv:.2f}%")

In [ ]:
dataset = {}      # dataset[h] = tabel lengkap
latih = {}        # latih[h] = baris dengan fitur lengkap + target
masa_depan = {}   # masa_depan[h] = baris untuk meramal

for h in range(1, cfg.HORIZON_MAKS + 1):
    hasil = ft.buat_fitur_lightgbm(df, kal_fitur, horizon=h, kolom_target=cfg.KOLOM_TARGET)
    l, m = ft.pisah_latih_dan_masa_depan(hasil)

    dataset[h] = hasil
    latih[h] = l
    masa_depan[h] = m

    print(f"h={h} | total origin={len(hasil)} | siap latih={len(l)} "
          f"| origin pertama={l.index.min().strftime('%Y-%m')} "
          f"| origin terakhir={l.index.max().strftime('%Y-%m')} "
          f"| baris masa depan={len(m)}")

In [ ]:
h = 3
print(f"Contoh dataset horizon h={h} (5 baris pertama yang siap latih):\n")
print(latih[h].head().round(3).T)

In [ ]:
# Uji pada beberapa origin dan semua horizon.
# Origin dipilih di bagian awal, tengah (masa COVID), dan akhir data.
origin_uji = [pd.Timestamp("2018-06-01"), pd.Timestamp("2020-03-01"),
              pd.Timestamp("2023-04-01"), pd.Timestamp("2025-12-01")]

semua_aman = True
for h in range(1, cfg.HORIZON_MAKS + 1):
    for origin in origin_uji:
        aman = ft.uji_kebocoran(df, kal_fitur, h, origin, cfg.KOLOM_TARGET)
        if not aman:
            semua_aman = False
            print(f"BOCOR: horizon={h}, origin={origin.strftime('%Y-%m')}")

if semua_aman:
    print(f"Semua uji lolos ({cfg.HORIZON_MAKS} horizon x {len(origin_uji)} origin): tidak ada kebocoran data.")

# uji yang sama pada target alternatif
aman_alt = ft.uji_kebocoran(df, kal_fitur, 3, pd.Timestamp("2023-04-01"), cfg.KOLOM_TARGET_ALT)
print("Uji target alternatif:", "aman" if aman_alt else "BOCOR")

In [ ]:
bulan_terakhir = df.index.max()
print("Bulan aktual terakhir :", bulan_terakhir.strftime("%Y-%m"))
print("Kalender sampai       :", kal_lengkap.index.max().strftime("%Y-%m"))
print()

for h in range(1, cfg.HORIZON_MAKS + 1):
    tgl_target = bulan_terakhir + pd.DateOffset(months=h)
    ada = tgl_target in kal_lengkap.index
    ada_baris = bulan_terakhir in masa_depan[h].index
    print(f"h={h} -> meramal {tgl_target.strftime('%Y-%m')} | kalender ada: {ada} | baris masa depan siap: {ada_baris}")

In [ ]:
# Korelasi fitur lag dengan target. Makin jauh horizon,
# informasi dari data terakhir (y_lag0) seharusnya makin lemah.
# Dihitung pada data pasca-COVID yang bersih saja.
baris = []
for h in range(1, cfg.HORIZON_MAKS + 1):
    d = latih[h]
    bersih = d[d["tercemar_covid"] == 0]
    baris.append({
        "horizon": h,
        "n_baris_bersih": len(bersih),
        "corr_y_lag0": bersih["y_lag0"].corr(bersih["y_target"]),
        "corr_y_tahun_lalu": bersih["y_tahun_lalu"].corr(bersih["y_target"]),
        "corr_roll_mean_12": bersih["roll_mean_12"].corr(bersih["y_target"]),
    })
print(pd.DataFrame(baris).round(3))

n_tercemar = sum((latih[h]["tercemar_covid"] == 1).sum() for h in latih)
n_total = sum(len(latih[h]) for h in latih)
print(f"\nBaris tercemar COVID: {n_tercemar} dari {n_total}")

In [ ]:
d = latih[3]
fig, ax = plt.subplots(figsize=(12, 3))
ax.scatter(d.index, d["tercemar_covid"], s=10, color="tomato")
ax.set_yticks([0, 1])
ax.set_yticklabels(["bersih", "tercemar"])
ax.set_title("Baris fitur yang tercemar COVID (contoh horizon h=3), sumbu x = origin")
plt.savefig(os.path.join(FOLDER_GAMBAR, "fitur_02_tercemar_covid.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Origin tercemar pertama :", d[d["tercemar_covid"] == 1].index.min().strftime("%Y-%m"))
print("Origin tercemar terakhir:", d[d["tercemar_covid"] == 1].index.max().strftime("%Y-%m"))

In [ ]:
# 1) regressor SARIMAX (sampai bulan kalender terakhir) + target
data_sarimax = exog.join(df[[cfg.KOLOM_TARGET, cfg.KOLOM_TARGET_ALT]], how="left")
data_sarimax.to_csv(os.path.join(FOLDER_PROCESSED, "data_sarimax.csv"))

# 2) dataset LightGBM per horizon
for h in range(1, cfg.HORIZON_MAKS + 1):
    latih[h].to_csv(os.path.join(FOLDER_PROCESSED, f"fitur_lgbm_h{h}.csv"))
    masa_depan[h].to_csv(os.path.join(FOLDER_PROCESSED, f"fitur_future_h{h}.csv"))

# 3) kalender fitur lengkap (dipakai backtest dan peramalan)
kal_fitur.to_csv(os.path.join(FOLDER_PROCESSED, "kalender_fitur.csv"))

print("File tersimpan:")
for nama in sorted(os.listdir(FOLDER_PROCESSED)):
    print(" -", nama)

## Ringkasan Hasil
- Jumlah fitur LightGBM: 15 (6 dari masa lalu, 9 dari kalender bulan target)
- Origin siap latih per horizon: h1=..., h6=...
- Uji kebocoran data: lolos / ada yang bocor
- Baris tercemar COVID: ... dari ...
- Horizon masa depan yang kalendernya sudah siap: ...
- Catatan untuk fase backtest: ...